# ฮองเรียน — ประลองโมเดลเสียงพากย์ไทยบน A100

ให้ทุกโมเดลพูดประโยคชุดเดียวกัน ด้วยเสียงอ้างอิงเดียวกัน แล้ววัด 3 อย่าง:
- **CER** — ถอดเสียงกลับด้วย Typhoon Whisper แล้วนับอักษรที่ผิด (ต่ำ = ออกเสียงถูก ไม่ตกหล่น)
- **ความเหมือนเสียงอ้างอิง** — WavLM speaker verification (สูง = โคลนเสียงได้เหมือน)
- **เวลาต่อประโยค**

แล้วฟังเทียบเองแบบไม่รู้ว่าเป็นโมเดลไหน (เซลล์สุดท้าย)

Runtime → **A100 GPU** → Run all · OpenJaiTTS ต้องขอสิทธิ์ที่หน้าโมเดลก่อน และใส่ token ไว้ใน Secrets ชื่อ `HF_TOKEN`
(ไม่มีก็ข้ามตัวนั้นไป ตัวอื่นยังทำงาน)

## 1. ประโยคทดสอบ (เขียนใหม่ เลียนแบบบทเรียนโปรแกรมมิ่ง: ศัพท์อังกฤษปน ตัวเลข สั้น/ยาว)

In [ ]:
SENTENCES = [
    "สวัสดีครับ วันนี้เราจะมาเรียนเรื่อง HTML กัน",
    "เปิดไฟล์ index.html ขึ้นมาในโปรแกรม VS Code ก่อนนะครับ",
    "ฟังก์ชันนี้รับชื่อเข้ามา แล้วคืนค่าเป็นข้อความทักทาย",
    "ตอนนี้หัวข้อเปลี่ยนเป็นสีแดงแล้ว",
    "เราจะใช้ CSS selector เลือกทุก element ที่มีคลาสชื่อ button",
    "ลองบันทึกไฟล์ แล้วกดรีเฟรชหน้าเว็บดูครับ",
    "ตัวแปร count เริ่มต้นที่ 0 แล้วเพิ่มขึ้นทีละ 1 ทุกครั้งที่วนลูป",
    "ถ้าค่าที่ได้มากกว่า 100 ให้แสดงคำว่า เกินกำหนด บนหน้าจอ",
    "อันนี้สำคัญมาก เพราะเราจะใช้มันซ้ำในโปรเจกต์ถัดไปอีกหลายครั้ง",
    "Git คือเครื่องมือที่ช่วยเก็บประวัติการแก้ไขโค้ดของเรา",
    "พิมพ์คำสั่ง git commit แล้วตามด้วยข้อความอธิบายสั้นๆ",
    "ในบทที่ 12 เราจะเชื่อมต่อฐานข้อมูล PostgreSQL ผ่าน Node.js",
    "ไม่ต้องกังวลถ้ายังไม่เข้าใจตอนนี้ เดี๋ยวเราจะกลับมาอธิบายอีกครั้งพร้อมตัวอย่างที่ชัดเจนขึ้น",
    "เว็บไซต์ส่วนใหญ่ในปี 2025 ใช้ JavaScript ทั้งฝั่ง frontend และ backend",
    "คลิกขวาที่หน้าเว็บ แล้วเลือก Inspect เพื่อเปิดเครื่องมือสำหรับนักพัฒนา",
    "เอาล่ะ พักกันสักครู่ แล้วเจอกันในบทเรียนหน้าครับ",
]
REF_TEXT = "สวัสดีครับ วันนี้เราจะมาเรียนรู้การสร้างเว็บไซต์ด้วยภาษา HTML และ CSS กันนะครับ"
OUT = "/content/bakeoff"
import os; os.makedirs(OUT, exist_ok=True)

## 2. ติดตั้ง

In [ ]:
!pip install -q omnivoice soundfile librosa jiwer
!pip install -q git+https://github.com/biodatlab/thonburian-tts.git
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. ตัวช่วย: วัดเวลา เก็บผล ถอดเสียง และวัดความเหมือนเสียง

In [ ]:
import time, json, traceback, re
import numpy as np, soundfile as sf, librosa, torch

try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN") or "")
except Exception:
    pass

RESULTS = {}  # model -> list of {"i", "wav", "seconds", "audio_seconds"}

def run_model(name, synth):
    """synth(text) -> (samples, sample_rate). A failure skips the model, not the run."""
    rows = []
    try:
        for i, text in enumerate(SENTENCES):
            started = time.time()
            audio, sr = synth(text)
            spent = time.time() - started
            path = f"{OUT}/{name}_{i:02d}.wav"
            sf.write(path, np.asarray(audio, dtype=np.float32).reshape(-1), sr)
            rows.append({"i": i, "wav": path, "seconds": spent, "audio_seconds": len(np.asarray(audio).reshape(-1)) / sr})
            print(f"  {name} {i+1}/{len(SENTENCES)} {spent:.1f}s", end="\r")
        RESULTS[name] = rows
        print(f"\n{name}: เสร็จ เฉลี่ย {np.mean([r['seconds'] for r in rows]):.2f}s/ประโยค")
    except Exception:
        print(f"{name}: ข้าม —"); traceback.print_exc(limit=2)

## 4. เสียงอ้างอิง (สร้างด้วย OmniVoice ให้ทุกโมเดลโคลนเสียงเดียวกัน)

In [ ]:
from omnivoice import OmniVoice
omni = OmniVoice.from_pretrained("hotdogs/omnivoice-thai").to("cuda:0")
if getattr(omni, "audio_tokenizer", None) is not None:
    omni.audio_tokenizer = omni.audio_tokenizer.to("cuda:0")
ref = np.asarray(omni.generate(text=REF_TEXT, instruct="female")[0]).reshape(-1)
REF_WAV = f"{OUT}/reference.wav"
sf.write(REF_WAV, ref, 24000)
print("reference", round(len(ref) / 24000, 1), "s")

## 5. โมเดลที่ประลอง

In [ ]:
# A. OmniVoice (ตัวที่ใช้อยู่) — โคลนจากเสียงอ้างอิง
prompt = omni.create_voice_clone_prompt(ref_audio=REF_WAV, ref_text=REF_TEXT)
run_model("omnivoice", lambda t: (omni.generate(text=t, voice_clone_prompt=prompt)[0], 24000))

In [ ]:
# B/C. ThonburianTTS และ OpenJaiTTS ใช้ไลบรารี flowtts ตัวเดียวกัน ต่างกันที่ไฟล์โมเดล
from flowtts.inference import FlowTTSPipeline, ModelConfig, AudioConfig

def flow_model(checkpoint, vocab):
    pipe = FlowTTSPipeline(
        ModelConfig(language="th", model_type="F5", checkpoint=checkpoint, vocab_file=vocab,
                    vocoder="vocos", device="cuda"),
        AudioConfig(silence_threshold=-45, cfg_strength=2.5, speed=1.0),
    )
    def synth(text):
        # The two model cards show different call styles; use whichever this version has.
        if hasattr(pipe, "generate"):
            return pipe.generate(reference_audio=REF_WAV, reference_text=REF_TEXT, gen_text=text)
        path = f"{OUT}/_tmp.wav"
        pipe(text=text, ref_voice=REF_WAV, ref_text=REF_TEXT, output_file=path)
        audio, sr = sf.read(path)
        return audio, sr
    return synth

run_model("thonburian", flow_model("hf://biodatlab/ThonburianTTS/megaF5/mega_f5_last.safetensors",
                                   "hf://biodatlab/ThonburianTTS/megaF5/mega_vocab.txt"))

In [ ]:
run_model("openjai", flow_model("hf://JTS-AI/OpenJaiTTS-v1.0/model.pt", "hf://JTS-AI/OpenJaiTTS-v1.0/vocab.txt"))

## 6. ให้คะแนน

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline, Wav2Vec2FeatureExtractor, WavLMForXVector
import jiwer

asr_id = "typhoon-ai/typhoon-whisper-turbo"
asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(asr_id, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
                                                      use_safetensors=True).to("cuda:0")
proc = AutoProcessor.from_pretrained(asr_id)
asr = pipeline("automatic-speech-recognition", model=asr_model, tokenizer=proc.tokenizer,
               feature_extractor=proc.feature_extractor, max_new_tokens=448, chunk_length_s=30,
               torch_dtype=torch.bfloat16, device="cuda:0")
sv_fe = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
sv = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to("cuda:0")

def embed(path):
    audio, _ = librosa.load(path, sr=16000)
    inputs = sv_fe([audio], sampling_rate=16000, return_tensors="pt", padding=True).to("cuda:0")
    with torch.no_grad():
        return torch.nn.functional.normalize(sv(**inputs).embeddings, dim=-1)[0]

def norm(text):
    return re.sub(r"[\s\.,!?'\"“”‘’()\-]", "", text.lower())

ref_emb = embed(REF_WAV)
SUMMARY = {}
for name, rows in RESULTS.items():
    for r in rows:
        heard = asr(r["wav"], generate_kwargs={"language": "th"})["text"]
        r["heard"] = heard
        r["cer"] = jiwer.cer(norm(SENTENCES[r["i"]]), norm(heard))
        r["sim"] = float(torch.dot(ref_emb, embed(r["wav"])))
    SUMMARY[name] = {
        "CER %": round(100 * np.mean([r["cer"] for r in rows]), 2),
        "ความเหมือนเสียง": round(float(np.mean([r["sim"] for r in rows])), 3),
        "วินาที/ประโยค": round(float(np.mean([r["seconds"] for r in rows])), 2),
        "ประโยคที่ CER>10%": sum(r["cer"] > 0.10 for r in rows),
    }
print(json.dumps(SUMMARY, ensure_ascii=False, indent=1))
json.dump({"summary": SUMMARY, "rows": RESULTS}, open(f"{OUT}/results.json", "w"), ensure_ascii=False, indent=1)

## 7. ฟังเทียบแบบไม่รู้ชื่อโมเดล (เฉลยอยู่ท้ายเซลล์ พับไว้)

In [ ]:
import random, base64
from IPython.display import HTML, display
names = list(RESULTS)
random.seed(7)
labels = {}
html = ["<style>td{padding:4px 8px} audio{height:30px}</style><table>"]
for i, text in enumerate(SENTENCES):
    order = names[:]; random.shuffle(order)
    cells = []
    for k, name in enumerate(order):
        labels.setdefault(i, {})["ABC"[k]] = name
        data = base64.b64encode(open(f"{OUT}/{name}_{i:02d}.wav", "rb").read()).decode()
        cells.append(f"<td>{'ABC'[k]}<br><audio controls src='data:audio/wav;base64,{data}'></audio></td>")
    html.append(f"<tr><td style='max-width:340px'>{i+1}. {text}</td>{''.join(cells)}</tr>")
html.append("</table>")
display(HTML("".join(html)))
json.dump(labels, open(f"{OUT}/blind_key.json", "w"), ensure_ascii=False)
print("เฉลย: /content/bakeoff/blind_key.json")